# Regex audit: subject contamination

Run the marker regex against the pre-cleanup backup and current SQLite snapshot. Review every match manually; do not use matches as automatic deletion instructions. Source grain is one row per `(subject, problem_id)`.


In [ ]:
import re
import sqlite3
from pathlib import Path

root = Path.cwd().resolve()
paths = {
    'pre-cleanup': root / 'etl/olimpiads_data_v2/olimpiads.before-subject-cleanup.sqlite3',
    'current': root / 'etl/olimpiads_data_v2/olimpiads.sqlite3',
}
pattern = re.compile(
    r'правов|юридич|уголовн|преступлен|международн.{0,30}гуманитарн|'
    r'репрессали|реторси|правонарушен|правоотношен|законодательств|'
    r'вооруж[её]нн.{0,20}конфликт|судебн.{0,20}инстанц|гражданск.{0,20}кодекс',
    re.IGNORECASE,
)

for label, path in paths.items():
    uri = path.resolve().as_uri() + '?mode=ro&immutable=1'
    with sqlite3.connect(uri, uri=True) as db:
        rows = db.execute('SELECT subject, problem_id, statement, classifier FROM problems').fetchall()
    matches = [row for row in rows if pattern.search((row[2] or '') + ' ' + (row[3] or ''))]
    print(label, len(matches))
    for subject, problem_id, statement, _ in matches:
        print(subject, problem_id, statement[:180].replace('\\n', ' '))


## Observed results

The pre-cleanup snapshot matched 8 rows: seven legal tasks, IDs 12627–12633, and one false positive, math problem 4447 about a detective. The current snapshot matches only that false positive. The regex can miss unrelated tasks that use different vocabulary.